In [ ]:
from langchain_neo4j import Neo4jGraph,Neo4jVector
from langchain_mistralai import ChatMistralAI,MistralAIEmbeddings
from langchain_text_splitters import TokenTextSplitter
from langchain_community.document_loaders import WikipediaLoader
from langchain_experimental.graph_transformers import LLMGraphTransformer
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
import os


C:\Users\Khan\AppData\Local\Temp\ipykernel_5124\1399665058.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WikipediaLoader
C:\Users\Khan\AppData\Local\Temp\ipykernel_5124\1399665058.py:5: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


In [2]:
load_dotenv()

url = os.getenv("NEO4J_URI")
usr_name = os.getenv("NEO4J_USERNAME")
usr_pwd = os.getenv("NEO4J_PASSWORD")
db = os.getenv("NEO4J_DATABASE")

In [ ]:
### Initializing Chat and Embedding Models
mistral_llm = ChatMistralAI(model="mistral-large-latest", temperature=0)
mistral_embeddings = MistralAIEmbeddings(model="mistral-embed")

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/mistralai/Mixtral-8x7B-v0.1/resolve/main/tokenizer.json
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/mistralai/Mixtral-8x7B-v0.1/resolve/main/tokenizer.json
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/mistralai/Mixtral-8x7B-v0.1/resolve/main/tokenizer.json
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/mistralai/Mixtral-8x7B-v0.1/resolve/main/tokenizer.json
Retrying in 8s [Retry 4/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/mistralai/Mixtral-8x7B-v0.1/resolve/main/tokenizer.json
Retrying in 8s [Retry 5/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/mistralai/Mixtral-8x7B-v0.1/resolve/main/tokenizer.json


In [ ]:
def initialize_graph_data(query: str, llm: mistral_llm, graph: Neo4jGraph):

    print("Starting The Process")
    raw_docs = WikipediaLoader(query= query, load_max_docs=10).load()
    chunks = TokenTextSplitter(chunk_size= 520, chunk_overlap= 60).split_documents(raw_docs)
    print("Data Loaded and Splitted into chunks")

    graph = Neo4jGraph(url= url, password= usr_pwd, username= usr_name, database= db)
    

    graph_transformer = LLMGraphTransformer(llm= llm)
    graph_docs = graph_transformer.convert_to_graph_documents(chunks)
    
    graph.add_graph_documents(
        graph_documents= graph_docs,
        base_entity_as_nodes=True, 
        include_source=True 
    )
    print("Data Added to Graph")

In [ ]:
def hybrid_retriever(embeddings: mistral_embeddings):
    
    retrieval_query = """
    MATCH (node)-[:MENTIONS]->(e:__Entity__)
    OPTIONAL MATCH (e)-[r:!MENTIONS]-(target:__Entity__)
    WITH node, score, collect(DISTINCT e.id + " -- " + type(r) + " --> " + target.id) AS graph_context
    RETURN 
        node.text AS text, 
        score, 
        { relationships: graph_context } AS metadata
    """
    graph_vecstr = Neo4jVector.from_existing_graph(
        embedding=embeddings,
        url=url,
        username=usr_name,
        password=usr_pwd,
        index_name="combined_graph_idx",
        node_label="Document",
        text_node_properties=["text"],
        embedding_node_property="embedding",
        retrieval_query=retrieval_query
    )

    retriever = graph_vecstr.as_retriever (search_kwargs={"k": 4})
    return retriever

In [ ]:
def content_formatting(docs):
    formatted = []
    for doc in docs:
        item = f"Document Snippet:\n{doc.page_content}\n"
        if "relationships" in doc.metadata and doc.metadata["relationships"]:
            relationship = doc.metadata["relationships"]
            item += "Graph Triples:\n" + "\n".join(relationship)
        formatted.append(item)
    return "\n\n---\n\n".join(formatted)